# TA-005C — Threshold Optimization: DenseNet-121 Augmentado

**Objetivo:** Encontrar el umbral óptimo por clase para maximizar F1-macro en DenseNet-121 aug (AUC test = 0.8902).

**Método:** Grid search sobre validation set → evaluar con umbrales óptimos en test set.

In [1]:
import json
import numpy as np
import pandas as pd
import pydicom
import wandb
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from pathlib import Path
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, classification_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

BASE          = Path(r'E:\Taller Integrador I\ModelosIA\Radiografias')
CHECKPOINTS   = Path(r'E:\Taller Integrador I\ModelosIA\modelos\checkpoints')
NOTEBOOKS_DIR = Path(r'E:\Taller Integrador I\ModelosIA\modelos\Notebooks')
VAL_DIR       = BASE / 'dataset_split' / 'val'
TEST_DIR      = BASE / 'dataset_split' / 'test'
MANIFESTS     = BASE / 'dataset_split' / 'manifests'

CLASSES      = ['alveolar_pattern', 'bronchial_pattern', 'pleural_effusion', 'cardiomegaly', 'no_finding']
NUM_CLASSES  = len(CLASSES)
BATCH_SIZE   = 16
DROPOUT      = 0.4
BASELINE_AUC = 0.8902
BASELINE_F1  = 0.6196

print('Config OK')

Device: cuda
Config OK


In [2]:
wandb.init(
    project='vetxray-cnn',
    entity='dbaylont1-antenor-orrego-private-university',
    name='TA-005C-Threshold-Optimization',
    config={
        'task': 'TA-005C',
        'model': 'DenseNet-121-augmented',
        'baseline_auc': BASELINE_AUC,
        'baseline_f1': BASELINE_F1,
        'threshold_default': 0.5,
        'threshold_search': '0.05 - 0.95 step 0.05'
    },
    tags=['DenseNet121', 'threshold', 'TA-005C', 'VetXRay']
)
print('W&B run:', wandb.run.url)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Pcuser\_netrc.
wandb: Currently logged in as: dbaylont1 (dbaylont1-antenor-orrego-private-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B run: https://wandb.ai/dbaylont1-antenor-orrego-private-university/vetxray-cnn/runs/n537ho0c


In [3]:
TRANSFORM_VAL = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class VetXRayDataset(Dataset):
    def __init__(self, df, transform, img_dir):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.img_dir   = img_dir
        self.labels    = self._build_labels()

    def _build_labels(self):
        labels = np.zeros((len(self.df), NUM_CLASSES), dtype=np.float32)
        for i, row in self.df.iterrows():
            for j, cls in enumerate(CLASSES):
                if isinstance(row['TAG'], str) and cls in row['TAG']:
                    labels[i, j] = 1.0
        return labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        dcm = pydicom.dcmread(str(self.img_dir / row['FileName']))
        arr = dcm.pixel_array.astype(np.float32)
        if hasattr(dcm, 'PhotometricInterpretation') and dcm.PhotometricInterpretation == 'MONOCHROME1':
            arr = arr.max() - arr
        p2, p98 = np.percentile(arr, 2), np.percentile(arr, 98)
        arr = np.clip(arr, p2, p98)
        arr = (arr - p2) / (p98 - p2 + 1e-8)
        img = Image.fromarray((arr * 255).astype(np.uint8)).resize((224, 224), Image.LANCZOS)
        rgb = Image.merge('RGB', [img, img, img])
        return self.transform(rgb), torch.tensor(self.labels[idx], dtype=torch.float32)

df_val  = pd.read_csv(MANIFESTS / 'val.csv')
df_test = pd.read_csv(MANIFESTS / 'test.csv')

val_loader  = DataLoader(VetXRayDataset(df_val,  TRANSFORM_VAL, VAL_DIR),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(VetXRayDataset(df_test, TRANSFORM_VAL, TEST_DIR),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
print(f'Val: {len(df_val)} | Test: {len(df_test)}')

Val: 1063 | Test: 940


In [4]:
model = models.densenet121(weights=None)
model.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(model.classifier.in_features, NUM_CLASSES))

best_files = sorted(CHECKPOINTS.glob('densenet_aug_best*.pth'),
                    key=lambda p: int(p.stem.split('ep')[-1]))
ckpt = torch.load(best_files[-1], map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model = model.to(DEVICE).eval()
print(f'Modelo cargado: {best_files[-1].name}')

Modelo cargado: densenet_aug_best_ep15.pth


In [5]:
def get_predictions(loader):
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            with torch.amp.autocast('cuda'):
                out = model(imgs)
            all_probs.append(torch.sigmoid(out).cpu().numpy())
            all_labels.append(labels.numpy())
    return np.vstack(all_probs), np.vstack(all_labels)

print('Obteniendo predicciones en val...')
val_probs,  val_labels  = get_predictions(val_loader)
print('Obteniendo predicciones en test...')
test_probs, test_labels = get_predictions(test_loader)
print('Predicciones OK')

Obteniendo predicciones en val...
Obteniendo predicciones en test...
Predicciones OK


In [6]:
thresholds_search = np.arange(0.05, 0.96, 0.05)
optimal_thresholds = {}

print('=== Optimización de umbrales (val set) ===')
for j, cls in enumerate(CLASSES):
    best_f1, best_thr = 0.0, 0.5
    for thr in thresholds_search:
        preds = (val_probs[:, j] >= thr).astype(int)
        f1 = f1_score(val_labels[:, j], preds, zero_division=0)
        if f1 > best_f1:
            best_f1  = f1
            best_thr = thr
    optimal_thresholds[cls] = round(float(best_thr), 2)
    print(f'  {cls:<22} umbral: {best_thr:.2f}  F1 val: {best_f1:.4f}')

print(f'\nUmbrales óptimos: {optimal_thresholds}')

=== Optimización de umbrales (val set) ===
  alveolar_pattern       umbral: 0.75  F1 val: 0.6063
  bronchial_pattern      umbral: 0.35  F1 val: 0.4122
  pleural_effusion       umbral: 0.90  F1 val: 0.7593
  cardiomegaly           umbral: 0.70  F1 val: 0.6272
  no_finding             umbral: 0.60  F1 val: 0.8139

Umbrales óptimos: {'alveolar_pattern': 0.75, 'bronchial_pattern': 0.35, 'pleural_effusion': 0.9, 'cardiomegaly': 0.7, 'no_finding': 0.6}


In [7]:
thr_array = np.array([optimal_thresholds[c] for c in CLASSES])

preds_default  = (test_probs >= 0.5).astype(int)
preds_optimized = (test_probs >= thr_array).astype(int)

auc_test   = roc_auc_score(test_labels, test_probs, average='macro')
f1_default = f1_score(test_labels, preds_default,   average='macro', zero_division=0)
f1_optim   = f1_score(test_labels, preds_optimized, average='macro', zero_division=0)
acc_default = accuracy_score(test_labels.flatten(), preds_default.flatten())
acc_optim   = accuracy_score(test_labels.flatten(), preds_optimized.flatten())

print('=== RESULTADOS TEST ===')
print(f'AUC macro:          {auc_test:.4f}  (sin cambio, umbral no afecta AUC)')
print(f'F1  umbral=0.5:     {f1_default:.4f}')
print(f'F1  umbral optimo:  {f1_optim:.4f}  (+{f1_optim - f1_default:.4f})')
print(f'Acc umbral=0.5:     {acc_default:.4f}')
print(f'Acc umbral optimo:  {acc_optim:.4f}')

print('\n=== F1 por clase ===')
f1_per_class_default = f1_score(test_labels, preds_default,   average=None, zero_division=0)
f1_per_class_optim   = f1_score(test_labels, preds_optimized, average=None, zero_division=0)
for cls, thr, f1d, f1o in zip(CLASSES, thr_array, f1_per_class_default, f1_per_class_optim):
    print(f'  {cls:<22} thr={thr:.2f}  F1: {f1d:.4f} → {f1o:.4f}  ({f1o-f1d:+.4f})')

=== RESULTADOS TEST ===
AUC macro:          0.8902  (sin cambio, umbral no afecta AUC)
F1  umbral=0.5:     0.6196
F1  umbral optimo:  0.6311  (+0.0116)
Acc umbral=0.5:     0.8594
Acc umbral optimo:  0.8738

=== F1 por clase ===
  alveolar_pattern       thr=0.75  F1: 0.5581 → 0.5694  (+0.0113)
  bronchial_pattern      thr=0.35  F1: 0.2817 → 0.3043  (+0.0227)
  pleural_effusion       thr=0.90  F1: 0.7552 → 0.7429  (-0.0124)
  cardiomegaly           thr=0.70  F1: 0.6801 → 0.7192  (+0.0391)
  no_finding             thr=0.60  F1: 0.8226 → 0.8198  (-0.0028)


In [8]:
wandb.log({
    'test_auc':          auc_test,
    'f1_default':        f1_default,
    'f1_optimized':      f1_optim,
    'f1_mejora':         f1_optim - f1_default,
    'acc_default':       acc_default,
    'acc_optimized':     acc_optim,
})

for cls, thr, f1d, f1o in zip(CLASSES, thr_array, f1_per_class_default, f1_per_class_optim):
    wandb.log({f'threshold/{cls}': thr, f'f1_optim/{cls}': f1o, f'f1_default/{cls}': f1d})

print('Métricas loggeadas en W&B')

Métricas loggeadas en W&B


In [9]:
results = {
    'model': 'densenet121_augmented',
    'task': 'TA-005C',
    'test_auc': round(auc_test, 4),
    'f1_default_threshold': round(f1_default, 4),
    'f1_optimized': round(f1_optim, 4),
    'f1_mejora': round(f1_optim - f1_default, 4),
    'acc_default': round(acc_default, 4),
    'acc_optimized': round(acc_optim, 4),
    'optimal_thresholds': optimal_thresholds,
    'f1_por_clase': {cls: {'default': round(float(f1d), 4), 'optimized': round(float(f1o), 4)}
                     for cls, f1d, f1o in zip(CLASSES, f1_per_class_default, f1_per_class_optim)}
}

out = NOTEBOOKS_DIR / 'densenet_aug_threshold_results.json'
with open(out, 'w') as f:
    json.dump(results, f, indent=4)

print(f'Resultados guardados: {out}')
print(f'\nResumen final:')
print(f'  AUC:  {auc_test:.4f}')
print(f'  F1:   {f1_default:.4f} → {f1_optim:.4f}  (+{f1_optim-f1_default:.4f})')
print(f'  Umbrales: {optimal_thresholds}')

wandb.finish()
print('TA-005C completado.')

Resultados guardados: E:\Taller Integrador I\ModelosIA\modelos\Notebooks\densenet_aug_threshold_results.json

Resumen final:
  AUC:  0.8902
  F1:   0.6196 → 0.6311  (+0.0116)
  Umbrales: {'alveolar_pattern': 0.75, 'bronchial_pattern': 0.35, 'pleural_effusion': 0.9, 'cardiomegaly': 0.7, 'no_finding': 0.6}


acc_default,▁
acc_optimized,▁
f1_default,▁
f1_default/alveolar_pattern,▁
f1_default/bronchial_pattern,▁
f1_default/cardiomegaly,▁
f1_default/no_finding,▁
f1_default/pleural_effusion,▁
f1_mejora,▁
f1_optim/alveolar_pattern,▁
+11,...


TA-005C completado.
